# 10 — SWOT vertical chain validation

**Scientific question.** What exactly is `PIXC height`, and which geophysical
corrections must be applied to place SWOT and ICESat-2 in one vertical frame?

**Inputs** PIXC granules (`data/raw/pixc_nova_kakhovka/`), one RiverSP Node granule
on the same cycle/pass, EGG2015, ua_2019z.asc.
**Outputs** `outputs/figure_data/Fig03_*`, `outputs/tables/vertical_reference_audit.csv`.
**Statistical unit** one RiverSP node. **Validation** independent-product sign test.


In [1]:
# --- provenance: environment, git SHA and input hashes -----------------------
import hashlib, json, subprocess, sys, platform
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))
import numpy as np, pandas as pd, geopandas as gpd, pyproj, rasterio, matplotlib
from swot_dnipro import config as CFG

np.random.seed(CFG.SEED)

def sha256(p, cap=64 * 1024 * 1024):
    p = Path(p)
    if not p.exists() or p.stat().st_size > cap:
        return f"(skipped: {'missing' if not p.exists() else 'too large'})"
    h = hashlib.sha256()
    with open(p, "rb") as fh:
        for c in iter(lambda: fh.read(1 << 20), b""):
            h.update(c)
    return h.hexdigest()[:16]

def git_sha():
    try:
        return subprocess.check_output(["git", "rev-parse", "--short", "HEAD"],
                                       cwd=CFG.ROOT, text=True).strip()
    except Exception:
        return "(not a git checkout)"

print("python      ", platform.python_version())
print("numpy       ", np.__version__)
print("pandas      ", pd.__version__)
print("geopandas   ", gpd.__version__)
print("matplotlib  ", matplotlib.__version__)
print("pyproj/PROJ ", pyproj.__version__, "/", pyproj.proj_version_str)
print("rasterio/GDAL", rasterio.__version__, "/", rasterio.__gdal_version__)
print("git SHA     ", git_sha())
print("seed        ", CFG.SEED)
print()
for p in [CFG.EGG2015_TIF, CFG.UA2019Z_ASC, CFG.CORRECTOR_BY_STATION,
          CFG.KHERSON_GAUGE_PARQUET, CFG.KHERSON_ATL13]:
    print(f"  {sha256(p)}  {Path(p).name}")


python       3.12.3
numpy        2.5.2
pandas       3.0.5
geopandas    1.1.4
matplotlib   3.11.1
pyproj/PROJ  3.7.2 / 9.5.1
rasterio/GDAL 1.5.1 / 3.12.4
git SHA      7d9580b
seed         42

  (skipped: too large)  egg_2015.tif
  297440243e2ed8d6  ua_2019z.asc
  92d9fd71a1c8da39  egg2015_to_evrf2019_by_station.csv
  3380b29a0d68016b  80805_yearbook.parquet
  1deb7b5cfd362bd7  kherson_atl13_pass_levels.parquet


## 1. Product definitions — read from the granules, not assumed

PIXC PDD **D-56411 Rev C p. 53**: `geoid`, `solid_earth_tide`, `load_tide_fes` and
`pole_tide` are each *"reported for reference but is not applied to the reported
height"*; `height` is *"Height of the pixel above the reference ellipsoid"*.

ATL03 ATBD **v007 p. 8**: *"ATL03 heights are in the tide-free system"*, and the
solid-Earth/load/pole tides are **already applied** upstream (pp. 20, 26).

In [2]:
import netCDF4 as nc
f = sorted(CFG.PIXC_DIR.glob("*.nc"))[0]
ds = nc.Dataset(str(f)); pc = ds.groups["pixel_cloud"]
for k in ["height", "geoid", "solid_earth_tide", "load_tide_fes", "pole_tide"]:
    v = pc.variables[k]
    print(f"### {k}")
    for a in ("long_name", "source", "comment"):
        if a in v.ncattrs():
            print("   ", a, ":", str(v.getncattr(a))[:220])
ds.close()

### height
    long_name : height above reference ellipsoid
    comment : Height of the pixel above the reference ellipsoid.
### geoid
    long_name : geoid height
    source : EGM2008 (Pavlis et al., 2012)
    comment : Geoid height above the reference ellipsoid with a correction to refer the value to the mean tide system, i.e. includes the permanent tide (zero frequency).  This value is reported for reference but is not applied to the 
### solid_earth_tide
    long_name : solid Earth tide height
    source : Cartwright and Taylor (1971) and Cartwright and Edden (1973)
    comment : Solid-Earth (body) tide height. The zero-frequency permanent tide component is not included.  This value is reported for reference but is not applied to the reported height.
### load_tide_fes
    long_name : geocentric load tide height (FES)
    source : FES2014b (Carrere et al., 2016)
    comment : Geocentric load tide height. The effect of the ocean tide loading of the Earth's crust. This value is report

## 2. Sign test against RiverSP (same cycle and pass)

The sign is established by agreement with an independently documented product,
never by numerical plausibility.

In [3]:
d = pd.read_csv(CFG.FIGDATA / "Fig03_pixc_riversp_sign_validation.csv")
display(d[["variant", "chain", "interpretation", "median_diff_m", "nmad_m", "n_nodes"]])
print("\nAccepted chain: h_SWOT = height - solid_earth_tide - load_tide_fes - pole_tide")

,variant,chain,interpretation,median_diff_m,nmad_m,n_nodes
0,A,height − (st+lt+pt),hypothesised / documented,-0.001002,0.031481,1023
1,B,height,no tide correction,0.072009,0.031439,1023
2,C,height + (st+lt+pt),wrong sign,0.145050,0.031705,1023
3,D,height − geoid,geoid blunder,-23.894687,0.334866,1023



Accepted chain: h_SWOT = height - solid_earth_tide - load_tide_fes - pole_tide


## 3. Permanent-tide systems

| quantity | system |
|---|---|
| ATL13 `ht_water_surf` | tide-free |
| SWOT `h` after §2 (`solid_earth_tide` excludes the permanent tide) | mean-tide crust |
| EGG2015, EVRF2019 (EPSG:9389) | zero-tide |

For crustal ellipsoidal heights zero-tide ≡ mean-tide, so **only ICESat-2 needs
harmonising**, via `tide_earth_free2mean = 0.06029 − 0.180873 sin²φ` (ATL03 ATBD p. 125).

In [4]:
print(pd.read_csv(CFG.TABLES / "vertical_reference_audit.csv").to_string(index=False)[:2600])
for name, lat in [("Kherson", 46.6238), ("Nova Kakhovka", 46.7754), ("Rozumivka", 47.7712)]:
    print(f"  free2mean({name:15s} phi={lat:.4f}) = {CFG.free2mean(lat):+.4f} m")

                      quantity reference_frame ellipsoid                                 definition_and_applied_corrections                     permanent_tide_system                                       source                      conversion_required
              SWOT PIXC height    WGS84 / ITRF     WGS84                               raw ellipsoidal, NO geophysical corr                         n/a (uncorrected)                   PIXC PDD D-56411 RevC p.53                                     none
         SWOT solid_earth_tide               -         -           reported, NOT applied; excludes zero-freq permanent tide                                         -                                PIXC PDD p.53                     subtract from height
SWOT load_tide_fes / pole_tide               -         -                                              reported, NOT applied                                         -                                PIXC PDD p.53                     subtract from